In [ ]:
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd 

from misc_utils import concordance_plot,auto_split_range,compound_argsort,single_agg,double_agg

from pathlib import Path

plt.rcParams['image.aspect'] = 'auto'
plt.rcParams['image.interpolation'] = 'none'

# Loading

In [ ]:
# !wget https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE268179
# !wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE268nnn/GSE268179/soft/

In [ ]:
# raw_path = "/home/bmb/haxx/working/ceisel_mumm/raws/sc/"
# ann_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
raw_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"
ann_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"

In [ ]:
## We need to isolate each set of files into a directory for a scanpy import 

# !mkdir {raw_path}/GSM8287434_RGC_24+48h_ranger
# !mv {raw_path}/GSM8287434_RGC_24+48h_*.gz {raw_path}/GSM8287434_RGC_24+48h_ranger

# !mkdir {raw_path}/GSM8287438_RGC_12+72h_ranger
# !mv {raw_path}/GSM8287438_RGC_12+72h_*.gz {raw_path}/GSM8287438_RGC_12+72h_ranger

In [ ]:
annotations = pd.read_csv(ann_path + "ceisel_adata_metadata.csv")
print(annotations.shape)
annotations.head()

In [ ]:
annotations.loc[annotations['Unnamed: 0'].duplicated()]

In [ ]:
# Annotration barcodes combine file name with barcode, we're stripping it out to match the count files

stripped_annotation_ids = [id.split("_")[2] for id in annotations['Cell.ID']]
stripped_annotation_ids
file_assignment = ["_".join(id.split("_")[:2]) for id in annotations['Cell.ID']]
annotations['Stripped.Cell.ID'] = stripped_annotation_ids
annotations['File.Assignment'] = file_assignment
annotations.set_index('Stripped.Cell.ID',inplace=True)

middle_annotations = annotations[annotations['File.Assignment'] == '24_48']
outer_annotations = annotations[annotations['File.Assignment'] == '12_72']
middle_annotations.head()
outer_annotations.head()


In [ ]:
# After stripping there are duplicate codes. This is theoretically fine since we can join each frame individually. 
annotations.iloc[annotations.index.duplicated()]

In [ ]:
middle_raw = sc.read_10x_mtx(
    raw_path + "/GSM8287434_RGC_24+48h_ranger",
    prefix="GSM8287434_RGC_24+48h_"
)
print(middle_raw.shape)
outer_raw = sc.read_10x_mtx(
    raw_path + "/GSM8287438_RGC_12+72h_ranger",
    prefix="GSM8287438_RGC_12+72h_"
)
print(outer_raw.shape)

In [ ]:
# Not all barcodes that are in the annotation file are also present in the raw file 
# Presumably annotation was created after original filtering

middle_ann_names = set(middle_annotations.index)
matching_middle = [id in middle_ann_names for id in middle_raw.obs_names]
print(f"middle: {np.sum(matching_middle)} out of {middle_raw.shape[0]}")
middle_raw.obs['annotated'] = matching_middle
outer_ann_names = set(outer_annotations.index)
matching_outer = [id in outer_ann_names for id in outer_raw.obs_names]
print(f"outer: {np.sum(matching_outer)} out of {outer_raw.shape[0]}")
outer_raw.obs['annotated'] = matching_outer


In [ ]:
# Retain only cells that are present in the annotation file
# Then join the annotations

middle_annotated = middle_raw[middle_raw.obs['annotated']]
outer_annotated = outer_raw[outer_raw.obs['annotated']]
middle_annotated.obs = middle_annotated.obs.join(middle_annotations)
outer_annotated.obs = outer_annotated.obs.join(outer_annotations)
middle_annotated.obs.head()
outer_annotated.obs.head()


In [ ]:
joint_raw = ad.concat([middle_annotated,outer_annotated])
joint_raw.shape

In [ ]:
# Dedupes the barcodes that are duplicated 
joint_raw.obs_names_make_unique()

In [ ]:
sc.write(ann_path + "joint_raw_new.h5ad",joint_raw)

# QC On Annotated Data

In [ ]:
ann_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"
joint_raw = sc.read_h5ad(ann_path + "joint_raw_new.h5ad")
joint_raw.shape


In [ ]:
sc.pp.calculate_qc_metrics(joint_raw, inplace=True)

In [ ]:
split_names = [n.split("-") for n in joint_raw.obs_names]
suffix = [n[-1] for n in split_names]

joint_raw.obs["library"] = suffix

plt.figure()
plt.title("Total Counts (All)")
plt.hist(joint_raw.obs["total_counts"],bins=np.arange(0,10000,100))
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Cells Per Library")
plt.hist(suffix)
plt.xlabel("Library")
plt.ylabel("Frequency")
plt.show()

plt.figure()
plt.title("Total Counts Split by Library")
for suffix in set(suffix):
    masked = joint_raw[joint_raw.obs["library"] == suffix]
    plt.hist(masked.obs["total_counts"],bins=np.arange(800,4000,100),alpha=0.3,label=suffix)
plt.xlabel("Counts Per Cell")
plt.ylabel("Frequency")
plt.legend()
plt.show()

# Retaining Signature Genes

#### Parthanatos

In [ ]:
parthanatos_signature_genes = ["Il1b","Aif1","App","Dlg4","Endog","F2","Grin2b","Icam1","Il1b","Map2k1","Mapk1","Mmp9","Parg","Parp1","Plat","S100b","Sirt1"]
parthanatos_signature_genes = [g.lower() for g in parthanatos_signature_genes]
parthanatos_signature_set = set(parthanatos_signature_genes)
parthanatos_signature_set

In [ ]:
parthanatos_present = [p for p in parthanatos_signature_set if p in set(joint_raw.var_names)]
parthanatos_missing = [p for p in parthanatos_signature_set if p not in set(joint_raw.var_names)]

print(parthanatos_present)
print(parthanatos_missing)


parthanatos_replacement_addons = [
    'parga','pargl',
    'dlg4a','dlg4b','dlg4b-1',
    'aif1l',
    'appa','appb',
    'grin2bb'
]

# F2 and s100b are present in the ann, but fall below even very conservative filtering thresholds. Removing. 
parthanatos_present.remove('f2')
parthanatos_present.remove('s100b')

print(joint_raw[:,'f2'].var)
print(joint_raw[:,'s100b'].var)

In [ ]:
# [g for g in joint_raw.var_names if "f2" in g[:3]]

# icam1: searched for the cd54 alias, searched or all icams and found only icam3
# s100b: questionably present, difficult to analogize? Letter soup present is t,a,w,v,u, and z
# f2 should theoretically be present? Not sure why it's not showing up. Omitting 

parthanatos_translated_signature = parthanatos_present + parthanatos_replacement_addons
parthanatos_translated_signature


#### Apoptosis

In [ ]:
apoptosis_signature_genes = ["Bax","Cycs","Akt1","Akt1","Bax","Bcl2l1","Birc2","Cycs","Ikbkb","Irak1","Pik3ca","Pik3r1","Pik3r2","Ppp3ca","Ppp3cb","Ppp3r1","Prkaca","Prkacb","Prkar1a","Prkar1b","Xiap"]
apoptosis_signature_genes = [g.lower() for g in apoptosis_signature_genes]
apoptosis_signature_set = set(apoptosis_signature_genes)

apoptosis_present = [p for p in apoptosis_signature_set if p in set(joint_raw.var_names)]
apoptosis_missing = [p for p in apoptosis_signature_set if p not in set(joint_raw.var_names)]

print(apoptosis_present)
print(apoptosis_missing)

In [ ]:
# Same as above, cycsa is gettinf filtered out, so I'm omitting. 
apoptosis_replacement_addons = [
    'baxb','baxa','baxa-1',
    'cycsb', # 'cycsa',
    'prkacaa','prkacab',
    'prkacba','prkacbb',
    'prkar1aa','prkar1ab','prkar1b',
    'ppp3r1b', 'ppp3r1a'
]

In [ ]:
# Manual replacement search 
print([g for g in joint_raw.var_names if "birc2" in g])

# baxa,baxb,baxa-1 for bax

apoptosis_translated_signature = apoptosis_present + apoptosis_replacement_addons
apoptosis_translated_signature


#### Necroptosis General(?)

In [ ]:
# These duplicates are an exact copy from the xls from the paper. Why? I don't know. The coefficients they claim appear duplicated so I don't think it's affecting their analysis?
necroptosis_signature_genes = ["Pgam5","Ripk3","Hmgb1","Dnm1l","Hmgb1","Mlkl","Pgam5","Ripk1","Ripk3","Mlkl","Ripk1"] 
necroptosis_signature_genes = [g.lower() for g in necroptosis_signature_genes]
necroptosis_signature_set = set(necroptosis_signature_genes)

necroptosis_present = [p for p in necroptosis_signature_set if p in set(joint_raw.var_names)]
necroptosis_missing = [p for p in necroptosis_signature_set if p not in set(joint_raw.var_names)]

print(necroptosis_present)
print(necroptosis_missing)


In [ ]:
print([g for g in joint_raw.var_names if "flj" in g])

# Mlkl appears wholly missing. The only alias I see online is flj34389, which is also absent (very few fljs generally)
necroptosis_replacement_addons = [
    'hmgb1a','hmgb1b',
    'ripk1l',
]

# HMGB1A & B are gregarious in development, we can mask them and see what happens

necroptosis_translated_signature = necroptosis_present + necroptosis_replacement_addons
necroptosis_translated_signature

In [ ]:
# necroptosis_translated_signature = ['pgam5', 'ripk3', 'dnm1l', 'ripk1l'] # Kicking out the dev

# Hand-curated instead, attr Ceisel
necroptosis_translated_signature = [
    'ripk1l','ripk3','fadd',
    'cyldb','cylda','cyldl',
    'igfbp2a','igfbp2b',
    'pgam5','tnfrsf1a','vdac1','vdac2','rbck1','tradd','dnm1l']

In [ ]:

population_marker_sets = {
    'cones': ['gnat2','arr3a','arr3b'],
    'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
    'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
    'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
    'rgc_secondary':['nrn1a','isl2b','islr2',],
    'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
    'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], 
    'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
    'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
    'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
    'pr_precursors':['tulp1a','opn6a','tmem244'],
    'erythrocytes':['hbbe2','prdx2','creg1'],
    # 'yama':['pou5f3','myca','mycb','klf17','sox2'],
}

all_cell_markers = [g for l in population_marker_sets.values() for g in l]
all_cell_markers

In [ ]:
for population in population_marker_sets:
    with open(f"population_markers.txt","a") as f:
        f.write(f"{population}: {str(",".join(population_marker_sets[population]))}\n")


In [ ]:
candidates = {}
for marker in all_cell_markers:
    candidates[marker] = [g for g in joint_raw.var_names if marker in g]
candidates

In [ ]:
print([g for g in joint_raw.var_names if "tag" in g])

# Filtering For Salience

## Finalized

In [ ]:
keepers = {
    'parthanatos':parthanatos_translated_signature,
    'necroptosis':necroptosis_translated_signature,
    'apoptosis':apoptosis_translated_signature,
    # 'parthanatos':[
    #     'mapk1', 'aif1l', 'dlg4a', 'appa', 'grin2bb', 'appb', 'parp1', 'plat',
    #     'endog', 'mmp9', 'parga', 'sirt1', 'map2k1', 'dlg4b', 'dlg4b-1', 'il1b','pargl'],
    # 'necroptosis':[
    #     'ripk3', 'tradd', 'cylda', 'fadd', 'pgam5', 'igfbp2a', 'cyldl',
    #     'ripk1l', 'igfbp2b', 'tnfrsf1a', 'vdac2', 'vdac1', 'cyldb', 'rbck1','dnm1l'],
    # 'apoptosis':[
    #     'pik3r1', 'prkar1b', 'baxb', 'prkar1aa', 'baxa', 'baxa-1', 'prkacaa',
    #     'cycsb', 'prkacbb', 'pik3r2', 'ppp3r1b', 'prkacab', 'ikbkb', 'irak1',
    #     'akt1', 'ppp3cb', 'xiap', 'ppp3r1a', 'prkar1ab', 'bcl2l1', 'birc2','ppp3ca', 'prkacba', 'pik3ca'],
    **population_marker_sets
}

In [ ]:
sc.pp.filter_genes(joint_raw, min_cells=1)
sc.pp.filter_cells(joint_raw, min_counts=1)

copy = joint_raw.copy()
sc.pp.log1p(copy)
sc.pp.highly_variable_genes(copy, n_top_genes=5000)

flat_keepers = set([g for v in keepers.values() for g in v]) # Flatten the above list
keeper_mask = np.zeros(copy.shape[1],dtype=bool)
keeper_mask[copy.var_names.isin(flat_keepers)] = True
# copy.var.loc[flat_keepers,'highly_variable'] = True

joint = joint_raw[:,copy.var["highly_variable"] | keeper_mask].copy()

# Let's annotate the genes per signature for the actual signatures
joint.var['parthanatos'] = [g in keepers['parthanatos'] for g in joint.var_names]
joint.var['necroptosis'] = [g in keepers['necroptosis'] for g in joint.var_names]
joint.var['apoptosis'] = [g in keepers['apoptosis'] for g in joint.var_names]


# sc.pp.downsample_counts(joint,counts_per_cell=1000)
sc.pp.normalize_total(joint,exclude_highly_expressed=True)
joint.layers['raw'] = joint.X.copy()
sc.pp.log1p(joint)

# Linear batch regression?
# Unfortunately linearly regressing out batch (or any other batch resolution idea) is probably bad because it would nuke a dangerouly large amount of the time variance. =/
# sc.pp.regress_out(joint, ['Batch'])

In [ ]:
joint.shape

# PCA And UMAP

In [ ]:
print("Computing PCA")
sc.pp.pca(joint, n_comps=50)
print("Computing neighbors")
sc.pp.neighbors(joint)
print("Computing UMAP")
sc.tl.umap(joint)
sc.pl.umap(joint)


In [ ]:
log_total_counts = np.log1p(joint.obs['total_counts'])
joint.obs['log_total_counts'] = log_total_counts
sc.pl.umap(joint, color=["log_total_counts"])
sc.pl.umap(joint, color=["library"])
sc.pl.umap(joint, color=["Batch"])

In [ ]:
time = [c.split("_")[0].strip("h") for c in joint.obs['Condition']]
condition = ["_".join(c.split("_")[1:]) for c in joint.obs['Condition']]
joint.obs['exp_condition'] = condition
joint.obs['time'] = np.array(time).astype(int)


In [ ]:
sc.pl.umap(joint, color=["time"])

In [ ]:
data_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"
sc.write(data_path + "joint_annotated_new_neighbors.h5ad",joint)

In [ ]:

plt.figure()
plt.title("Log total counts vs time")
plt.boxplot(
    [joint.obs['log_total_counts'][joint.obs['time'] == t] for t in [12,24,48,72]],
    showfliers=False,
)
plt.xticks(
    [1,2,3,4],
    ["12h","24h","48h","72h"]
)
plt.ylabel("Log total counts")
plt.show()

# fig,(ax0,ax1) = plt.subplots(1,2)
# ctrl_mask = np.array(joint.obs['exp_condition'] == 'Cntr')
# ax0.boxplot(
#     []

# Cell Type Annotation

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "joint_annotated_new_neighbors.h5ad")

In [ ]:
sc.tl.leiden(data)
sc.pl.umap(data, color=['leiden'],legend_loc='on data')

In [ ]:
# As before
# population_marker_sets = {
#     'cones': ['gnat2','arr3a','arr3b'],
#     'rods': ['rho','gnat1','gngt1','sagb','pde6a','pde6b','nrl'],
#     'muller_glia':['rlbp1a','rx1','crabp1a','gpr37a',],
#     'rgcs':['rbpms2b','isl2b','uchl1','rbpms2a','pou4f2'],
#     'rgc_secondary':['nrn1a','isl2b','islr2',],
#     'microglia':['mfap4','mpeg1.1','csf1ra','spi1b'],
#     'bipolar_cells':['nrn1lb','cabp5a','vsx1','calb2b','vamp1','rs1a','syt5b',], 
#     'amacrine_cells':['pax6b','slc6a1b','pax10','nhlh2','pvalb6'], # pax6a got filtered
#     'horizontal_cells':['onecutl','tnr','rem1','CR361564.1','cx52.9','ompa','opn9'], # onecut ambiguous, cx55.5, slc4a5b seem fully missing from the index
#     'undifferentiated':['sox2','olig2','hes6','hmgn2','hmga1a','atoh7'],
#     'pr_precursors':['tulp1a','opn6a','tmem244'],
#     'erythrocytes':['hbbe2','prdx2','creg1'],
#     'yama':['pou5f3','myca','mycb','klf17','sox2']
# }


In [ ]:
sc.pl.dotplot(data,population_marker_sets,groupby="leiden",figsize=(10,8))

In [ ]:
mapping = {
    k:k for k in data.obs['leiden'].unique()
}

# THIS MAPPING WILL SHIFT PER EVALUATION OF LEIDEN, BECAUSE LEIDEN IS STOCHASTIC. RE-DO PER RUN USING THE DOTPLOT


reverse_mapping = {
    'Bipolar cells':[0,1,2,3,5,9,15,20,],
    'Horizontal cells':[10,12,28],
    'Cones':[8,13,19,],
    'Rods':[23,],
    'Muller glia':[11,],
    'RGCs':[18,],
    'Microglia':[26,],
    'Amacrine cells':[4,16],
    'Erythrocytes':[24,],
    'injury':[29,],
}

# reverse_mapping = {
#     'Bipolar cells':[0,2,3,4,7,11,12,14,19,],
#     'Horizontal cells':[1,30],
#     'Cones':[5,10,18,],
#     'Rods':[24,],
#     'Muller glia':[9,],
#     'RGCs':[21,],
#     'Microglia':[29,],
#     'Amacrine cells':[6,15,20,],
#     'Erythrocytes':[27,],
#     'injury':[31,],
# }

mapping |= {
    str(v):k for k,l in reverse_mapping.items() for v in l
} 



# mapping
# Create a new column with renamed clusters
data.obs['cell_type'] = data.obs['leiden'].map(mapping).astype('category')


In [ ]:
sc.pl.dotplot(data,population_marker_sets,groupby="cell_type",figsize=(10,8))

In [ ]:
sc.pl.umap(data,color='cell_type',legend_loc='on data')

In [ ]:
sc.tl.rank_genes_groups(data, 'cell_type')
sc.pl.rank_genes_groups(data, n_genes=25)

In [ ]:
sc.write(data_annotated_path + "full_annotations_leiden_new.h5ad",data)

# Scores

In [ ]:
data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_new.h5ad")

In [ ]:
# A small subpopulation of cells clusters out only within a single time point. 
# This is analytically interesting, but prevents clean comparisons between timepoints for other purposes, so we are going to mask these out for now
data = data[data.obs['cell_type'] != 'injury']

In [ ]:
parthanatos_mask = np.array(data.var['parthanatos'])
apoptosis_mask = np.array(data.var['apoptosis'])
necroptosis_mask = np.array(data.var['necroptosis'])

parthanatos_signature = data.var_names[parthanatos_mask]
apoptosis_signature = data.var_names[apoptosis_mask]
necroptosis_signature = data.var_names[necroptosis_mask]

print(parthanatos_signature)
print(necroptosis_signature)
print(apoptosis_signature)

sc.tl.score_genes(data,parthanatos_signature,score_name="parthanatos")
sc.tl.score_genes(data,necroptosis_signature,score_name="necroptosis")
sc.tl.score_genes(data,apoptosis_signature,score_name="apoptosis")

In [ ]:
# from misc_utils import auto_split_range

# fig,axes = plt.subplots(1,3,figsize=(25,5))
# axes = axes.flatten()

# sc.pl.umap(data,color='parthanatos',ax=axes[0],show=False,**auto_split_range(data.obs['parthanatos']))
# sc.pl.umap(data,color='necroptosis',ax=axes[1],show=False,**auto_split_range(data.obs['necroptosis']))
# sc.pl.umap(data,color='apoptosis',ax=axes[2],show=False,**auto_split_range(data.obs['apoptosis']))

# fig.show()

from misc_utils import auto_split_range

injury_mask = np.array(data.obs['exp_condition'] == 'Mtz')

fig,axes = plt.subplots(2,3,figsize=(25,10))

sc.pl.umap(data[injury_mask],color='parthanatos',ax=axes[0][0],show=False,**auto_split_range(data.obs['parthanatos']))
sc.pl.umap(data[injury_mask],color='necroptosis',ax=axes[0][1],show=False,**auto_split_range(data.obs['necroptosis']))
sc.pl.umap(data[injury_mask],color='apoptosis',ax=axes[0][2],show=False,**auto_split_range(data.obs['apoptosis']))

sc.pl.umap(data[~injury_mask],color='parthanatos',ax=axes[1][0],show=False,**auto_split_range(data.obs['parthanatos']))
sc.pl.umap(data[~injury_mask],color='necroptosis',ax=axes[1][1],show=False,**auto_split_range(data.obs['necroptosis']))
sc.pl.umap(data[~injury_mask],color='apoptosis',ax=axes[1][2],show=False,**auto_split_range(data.obs['apoptosis']))

fig.show()


In [ ]:
fig,axes = plt.subplots(1,2,figsize=(10,5))
sc.pl.dotplot(data[data.obs['exp_condition'] == 'Mtz'],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",ax=axes[0],show=False)
sc.pl.dotplot(data[data.obs['exp_condition'] != 'Mtz'],['parthanatos','necroptosis','apoptosis'],groupby="cell_type",ax=axes[1],show=False)
axes[0].set_title("Injury")
axes[1].set_title("Control")
fig.tight_layout()
fig.show()

# data.obs['time'] = data.obs['time'].astype(str)


In [ ]:
sc.pl.umap(data,color='time')

fig,axes = plt.subplots(2,4,figsize=(20,10))
legend_loc = 'none'
for t,ax in zip([12,24,48,72],axes[0]):
    legend_loc = 'on data' if t == 12 else 'none'
    sub = data[(data.obs['time'] == t) & (data.obs['exp_condition'] == 'Mtz')]
    sc.pl.umap(sub,color='cell_type',legend_loc=legend_loc,ax=ax,show=False)
    ax.set_title(f"Injury {t}h")
for t,ax in zip([12,24,48,72],axes[1]):
    sub = data[(data.obs['time'] == t) & (data.obs['exp_condition'] != 'Mtz')]
    sc.pl.umap(sub,color='cell_type',legend_loc=legend_loc,ax=ax,show=False)
    ax.set_title(f"Control {t}h")
fig.tight_layout()
fig.show()


# Miscellaneous Scratch Work

In [ ]:
# It costs us nothing to include this work, but it is unrelated to the core analysis and is presented a) with no warranty, express or implied, and b) purely for historical purposes

## Manually Sanity-Checking Intra-Cell Type Effects

In [ ]:
# sc.pl.dotplot(data,parthanatos_signature,groupby="cell_type")
rgc_subset = data[data.obs['cell_type'] == 'RGCs',parthanatos_signature]
injury_mask = np.array(rgc_subset.obs['exp_condition'] == 'Mtz')
time_sort_injury = np.argsort(rgc_subset[injury_mask].obs['time'].astype(int))
time_sort_control = np.argsort(rgc_subset[~injury_mask].obs['time'].astype(int))

rgc_matrix = rgc_subset.X.toarray()
injury_matrix = rgc_matrix[injury_mask][time_sort_injury]
control_matrix = rgc_matrix[~injury_mask][time_sort_control]

fig,(ax0,ax1) = plt.subplots(1,2)
fig.suptitle("RGCs only")
ax0.set_title("Injury")
im0 = ax0.imshow(
    injury_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
ax0.set_xticks(np.arange(injury_matrix.shape[1]),labels=parthanatos_signature,rotation=90)

ax1.set_title("Control")
im1 = ax1.imshow(
    control_matrix,
    cmap='viridis',
    aspect='auto',
    interpolation='nearest',vmin=0,vmax=1.5
)
ax1.set_xticks(np.arange(control_matrix.shape[1]),labels=parthanatos_signature,rotation=90)
plt.colorbar(im0,ax=ax0)
plt.colorbar(im1,ax=ax1)
fig.tight_layout()
fig.show()

In [ ]:
directional_exp = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else -1))
cell_type_one_hot = pd.get_dummies(data.obs['cell_type'])
leiden_one_hot = pd.get_dummies(data.obs['leiden'])

directional_cell_type = [cell_type_one_hot.iloc[:,i].astype(int) * directional_exp for i in range(cell_type_one_hot.shape[1])]
directional_cell_type = np.array(directional_cell_type).T

directional_leiden = [leiden_one_hot.iloc[:,i].astype(int) * directional_exp for i in range(leiden_one_hot.shape[1])]
directional_leiden = np.array(directional_leiden).T

concordance_plot(
    directional_cell_type,
    data.obsm['X_pca'],
    metric='correlation',
    title="Within-group Injury Correlation",
    label_1="Cell Type",
    label_2="PC",
    ticks_1=cell_type_one_hot.columns,
)


concordance_plot(
    directional_leiden,
    data.obsm['X_pca'],
    metric='correlation',
    title="Within-group Injury Correlation",
    label_1="Leiden",
    label_2="PC",
    ticks_1=leiden_one_hot.columns,
    figsize=(15,15)
)


concordance_plot(
    cell_type_one_hot,
    data.obsm['X_pca'],
    ticks_1=cell_type_one_hot.columns,
    title="PCs vs Pure Cell Types",
    label_1="Cell Type",
    label_2="PC",
)



In [ ]:
time_one_hot = pd.get_dummies(data.obs['time'].astype(str))
concordance_plot(
    time_one_hot,
    data.obsm['X_pca'],
    ticks_1=time_one_hot.columns,
    title="PCs vs Time, Categorical",
    label_1="Time",
    label_2="PC",
)




In [ ]:
from misc_utils import auto_split_range

fig,axes = plt.subplots(5,10,figsize=(20,10))
axes = axes.flatten()
for i,ax in enumerate(axes):
    ax.scatter(
        data.obsm['X_umap'][:,0],
        data.obsm['X_umap'][:,1],
        c=data.obsm['X_pca'][:,i],
        **auto_split_range(data.obsm['X_pca'][:,i]),
        s=1,
    )
    ax.set_title(f"PC {i}")
    # data.obs['pc_tmp'] = data.obsm['X_pca'][:,i]
    # sc.pl.umap(data,color='pc_tmp',ax=ax,show=False,**auto_split_range(data.obs['pc_tmp']))
fig.tight_layout()
fig.show()


In [ ]:
# data.obs['exp_tmp'] = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else 0)).astype(int)
sc.pl.umap(data,color='exp_condition')

## Diverging Heatmap

In [ ]:
n_types = len(data.obs['cell_type'].cat.categories)
n_types

In [ ]:

def diverging_heatmap_order(exp,time):
    exp = [-1 if x=="Mtz" else 1 for x in exp]
    time = [e*t for e,t in zip(exp,time)]
    heatmap_order = np.argsort(np.array(time))
    return heatmap_order


fig,axes = plt.subplots(4,5,figsize=(25,15))
for i,cell_type in enumerate(data.obs['cell_type'].cat.categories):
    print(cell_type)
    ax = axes.flatten()[i]
    subset = data[data.obs['cell_type'] == cell_type]
    heatmap_order = diverging_heatmap_order(subset.obs['exp_condition'],subset.obs['time'])

    local_pcs = subset.obsm['X_pca']
    local_pc_sort = single_agg(local_pcs.T)
    local_pcs = local_pcs[heatmap_order].T[local_pc_sort].T

    ax.imshow(
        local_pcs,
        **auto_split_range(local_pcs)
    )
    ax.set_xticks(np.arange(local_pcs.shape[1]),np.arange(local_pcs.shape[1])[local_pc_sort],rotation=90)
    ax.set_title(cell_type)
fig.tight_layout()
fig.show()

    # plt.figure()
    # plt.imshow(
    #     np.array(pos_subset.X.todense())[heatmap_order],
    #     # **auto_split_range(data.obsm['X_pca'])
    # )
    # plt.show()


## Parallel Heatmap


In [ ]:
# fig,axes = plt.subplots(4,5,figsize=(25,15))

for i,cell_type in enumerate(data.obs['cell_type'].cat.categories):
    # ax = axes.flatten()[i]
    subset = data[data.obs['cell_type'] == cell_type]
    injury_subset = subset[subset.obs['exp_condition'] == 'Mtz']
    control_subset = subset[subset.obs['exp_condition'] != 'Mtz']
    injury_time_sort = np.argsort(injury_subset.obs['time'])
    control_time_sort = np.argsort(control_subset.obs['time'])

    local_pc_sort = single_agg(injury_subset.obsm['X_pca'].T)

    fig,(ax0,ax0_bar,ax1,ax1_bar) = plt.subplots(1,4,width_ratios=(20,1,20,1),figsize=(15,5))
    fig.suptitle(cell_type)
    ax0.imshow(
        injury_subset.obsm['X_pca'][injury_time_sort].T[local_pc_sort].T,
        **auto_split_range(injury_subset.obsm['X_pca'])
    )
    ax0.set_xticks(np.arange(injury_subset.obsm['X_pca'].shape[1]),np.arange(injury_subset.obsm['X_pca'].shape[1])[local_pc_sort],rotation=90)
    ax0_bar.imshow(
        np.array(injury_subset.obs['time'][injury_time_sort]).reshape(-1,1),
    )

    ax1.imshow(
        control_subset.obsm['X_pca'][control_time_sort].T[local_pc_sort].T,
        **auto_split_range(control_subset.obsm['X_pca']),
    )
    ax1.set_xticks(np.arange(control_subset.obsm['X_pca'].shape[1]),np.arange(control_subset.obsm['X_pca'].shape[1])[local_pc_sort],rotation=90)
    ax1_bar.imshow(
        np.array(control_subset.obs['time'][control_time_sort]).reshape(-1,1),
    )

    fig.tight_layout()
    fig.show()

## Other

In [ ]:
leiden_one_hot = pd.get_dummies(data.obs['leiden'])
cell_type_one_hot = pd.get_dummies(data.obs['cell_type'])
exp_one_hot = np.array(data.obs['exp_condition'].map(lambda x: 1 if "Mtz" in x else -1)).reshape(-1,1)
concordance_plot(
    exp_one_hot,
    cell_type_one_hot,
    ticks_2=cell_type_one_hot.columns,
)
concordance_plot(
    exp_one_hot,
    leiden_one_hot,
)

In [ ]:
# # exp = data.obs['exp_condition']
# # time = data.obs['time']
# # exp = [-1 if x=="Mtz" else 1 for x in exp]
# # time = [exp*t for t in time]
# diverging_heatmap_order(data.obs['exp_condition'],data.obs['time'])


In [ ]:
# lol don't do this, just checking what paper did

mg_subset = data[data.obs['cell_type'] == 'Muller glia'].copy()

sc.tl.rank_genes_groups(mg_subset, 'exp_condition')
sc.pl.rank_genes_groups(mg_subset, n_genes=20)







In [ ]:
from scipy.cluster.hierarchy import dendrogram,linkage

def single_agg(mtx,metric='cosine',method='average'):
    return dendrogram(linkage(mtx, metric=metric, method=method), no_plot=True)['leaves']

def double_agg(mtx,metric='cosine',method='average'):
    row_agg = dendrogram(linkage(mtx, metric=metric, method=method), no_plot=True)['leaves']
    col_agg = dendrogram(linkage(mtx.T, metric=metric, method=method), no_plot=True)['leaves']
    return mtx[row_agg].T[col_agg].T


In [ ]:
data.obs['pc_tmp'] = data.obsm['X_pca'][:,4]
injury_mask = data.obs['exp_condition'] == "Mtz"
sc.pl.umap(data[injury_mask],color='pc_tmp',show=False,**auto_split_range(data.obs['pc_tmp']))
sc.pl.umap(data[~injury_mask],color='pc_tmp',show=False,**auto_split_range(data.obs['pc_tmp']))

In [ ]:
injury_mask = data.obs['exp_condition'] == "Mtz"
time_masks = [data.obs['time'] == t for t in [12,24,48,72]]

for i in range(40):
    ct_mask = data.obs['cell_type'] == "Bipolar cells"

    plt.figure()
    plt.title(f"PC {i}")

    plt.violinplot(
        [
            data.obsm['X_pca'][:,i][ct_mask & injury_mask & time_masks[j]]
            for j in range(4)
        ],
        positions=np.arange(4)-.15,widths=0.3
    )        
    plt.violinplot(
        [
            data.obsm['X_pca'][:,i][ct_mask & ~injury_mask & time_masks[j]]
            for j in range(4)
        ],
        positions=np.arange(4)+.15,widths=0.3
    )
    plt.show()



